# Packages import

In [20]:
import yaml
import os
import re
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# Apollo Scraper

In [21]:
with open("conifg.yaml","r", encoding = "UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config["credentials"]['user']
password = config["credentials"]['password']
credentials = HTTPBasicAuth(username, password)

In [22]:
url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252681&okres=2"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)

200


In [23]:
page_dom = BeautifulSoup(response.text, "html.parser")

In [24]:
groupe = page_dom.select_one("div.grupa").get_text(strip=True)
print(groupe)

ZICSS1-1211


In [25]:
classes_tag = page_dom.select_one("table")
with open("temp.html","w",encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html",encoding="UTF-8", header = 0)[0]
os.remove("temp.html")

In [26]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])]

In [27]:
classes[['Day','Start time','hyphen','End time','Duration']] = classes['Dzień, godzina'].str.split(' ',expand = True)

In [28]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [29]:
classes = classes.drop(['Dzień, godzina','hyphen'], axis = 1)

In [30]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.)*",
    r"\1",
    regex=True
)

In [31]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")

In [32]:
classes.to_csv(f"schedules/{groupe}.csv")